In [14]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

def load_and_clean(filename):
    return np.loadtxt(filename, skiprows=2)

data_x = load_and_clean('x24x24.txt')
data_y = load_and_clean('y24x24.txt')
data_z = load_and_clean('z24x24.txt')
full_data = np.vstack([data_x, data_y, data_z])

X = full_data[:, :576]
y = full_data[:, 578]


X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=67,
    stratify=y           # photos of a person will be in both sets
)

encoder = OneHotEncoder(sparse_output=False)
y_train = encoder.fit_transform(y_train.reshape(-1, 1))

In [57]:
learning_rate = 0.1
epochs = 700

def params_initialization():
    params = {
        #hidden layer - weight and bias
        "W1": np.random.randn(576, 128) * np.sqrt(2.0 / 576),
        "b1": np.zeros((1, 128)),

        #output layer - weight and bias
        "W2": np.random.randn(128, 48) * np.sqrt(2.0 / 128),
        "b2": np.zeros((1, 48))
    }

    return params


In [16]:
def forward_propagation(X, W1, b1, W2, b2):
    Z1 = np.dot(X, W1) + b1
    A1 = np.maximum(Z1, 0) # ReLU

    Z2 = np.dot(A1, W2) + b2

    exp_Z2 = np.exp(Z2 - np.max(Z2, axis=1, keepdims=True))
    A2 = exp_Z2 / np.sum(exp_Z2, axis=1, keepdims=True)

    data = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}

    return A2, data

In [17]:
def cross_entropy(prediction, target):
    sum_of_loss = np.sum(target * np.log(prediction + 1e-8), axis=1)

    return -np.mean(sum_of_loss)

In [18]:
def relu_derivative(Z):
    return (Z > 0).astype(float)

In [19]:
def backpropagation(X, target, data, params):
    m = X.shape[0]

    A1 = data['A1']
    A2 = data['A2']
    Z1 = data['Z1']
    W2 = params['W2']

    dZ2 = A2 - target

    dW2 = (1 / m) * np.dot(A1.T, dZ2)
    db2 = (1 / m) * np.sum(dZ2, axis=0, keepdims=True)

    dZ1 = np.dot(dZ2, W2.T) * relu_derivative(Z1)

    dW1 = (1 / m) * np.dot(X.T, dZ1)
    db1 = (1 / m) * np.sum(dZ1, axis=0, keepdims=True)

    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}



In [44]:
def params_actualization(params, gradients):
    params['W1'] -= learning_rate * gradients['dW1']
    params['b1'] -= learning_rate * gradients['db1']
    params['W2'] -= learning_rate * gradients['dW2']
    params['b2'] -= learning_rate * gradients['db2']

    return params

In [58]:
params = params_initialization()

for epoch in range(epochs):
    A2, data = forward_propagation(X_train, params['W1'], params['b1'], params['W2'], params['b2'])

    loss = cross_entropy(A2, y_train)
    print(loss)
    gradients = backpropagation(X_train, y_train, data, params)

    params = params_actualization(params, gradients)

4.172140882498809
3.941420906132971
3.8906743728346065
3.860204523872089
3.8371257080730743
3.8179895651381326
3.800951203207254
3.7852910249695078
3.771090174529605
3.7577830288377547
3.7450235721588854
3.732616964976243
3.720599435547852
3.708848329110331
3.697381811531
3.6861043431831493
3.674959848504209
3.66389247714463
3.6528274462684287
3.6418016669073885
3.630846405427403
3.6198786980527906
3.608832949822541
3.5977513319979666
3.58663011753879
3.575460221956062
3.5642544574369763
3.5530246226167352
3.5418244668062684
3.5306115801791007
3.519423622667984
3.5082115583958733
3.496936467790635
3.485626355320222
3.4742727368493704
3.462874786957157
3.451386538305606
3.4398524374952215
3.428304815856651
3.416711113714014
3.405118750249252
3.393486878548991
3.381810737941675
3.370059581509492
3.358244335626434
3.346361466505379
3.3344535626127145
3.322547725494244
3.310585275412681
3.298577407061561
3.2865307550942418
3.274441959106707
3.2622970134447433
3.2501458116205186
3.237966345

In [32]:
def predict(X, params, threshold=0.5):
    A2, _ = forward_propagation(X, params['W1'], params['b1'], params['W2'], params['b2'])

    predictions = np.argmax(A2, axis=1)

    max_probs = np.max(A2, axis=1)

    final_predictions = np.where(max_probs > threshold, predictions, -1)

    return final_predictions

In [61]:
y_pred_val = predict(X_val, params)

accuracy = np.mean(y_pred_val == y_val)

print(f"With 50%: {accuracy * 100:.2f}%")

With 50%: 51.94%


In [62]:
A2_test, _ = forward_propagation(X_val, params['W1'], params['b1'], params['W2'], params['b2'])
simple_predictions = np.argmax(A2_test, axis=1)
simple_accuracy = np.mean(simple_predictions == y_val)
print(f"Without 50%: {simple_accuracy * 100:.2f}%")

Without 50%: 71.91%
